In [1]:
#@title 0. Instalar dependencias y montar Drive
from google.colab import drive
drive.mount('/content/drive')

# Instalar detectron2 desde fuente
import sys, os, distutils.core

# Detectron2 (fast local install)
!git clone -q 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install -q {' '.join([f"'{x}'" for x in dist.install_requires])}
import sys, os
sys.path.insert(0, os.path.abspath('./detectron2'))

# Otros paquetes
!pip install -q rasterio pyproj fiona matplotlib albumentations tqdm pycocotools pandas scikit-image


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 90.9 MB/s eta 0:00:00


In [2]:
#@title 1. Rutas, constantes, semillas, bins de tamaño
import os, json, random, math
from pathlib import Path
import numpy as np
import torch
import cv2
import rasterio
from tqdm import tqdm

# ----- BASE y rutas (ajusta si tu estructura difiere) -----
BASE = Path("/content/drive/MyDrive/juniper_mapper/JuniperMapper")

# Imágenes PI
PI_IMG_DIR = str(BASE/"Photo_Interpretation_Data")
FW_IMG_DIR = str(BASE/"Field_Work_Data"/"External_Val_Data"/"Images")

# COCO instancia — PI (train/val/test) y FW (test)
PI_COCO_TRAIN_JSON = str(BASE/"Photo_Interpretation_Data"/"Train"/"Annotations"/"Train.json")
PI_COCO_VAL_JSON   = str(BASE/"Photo_Interpretation_Data"/"Val"/"Annotations"/"Val.json")
PI_COCO_TEST_JSON  = str(BASE/"Photo_Interpretation_Data"/"Test"/"Annotations"/"Test.json")
FW_COCO_TEST_JSON  = str(BASE/"Field_Work_Data"/"External_Val_Data"/"Annotations"/"FW.json")

# Directorios de imágenes PI (siguiendo tu convención)
PI_TRAIN_IMG_DIR = str(BASE/"Photo_Interpretation_Data"/"Train"/"Images")
PI_VAL_IMG_DIR   = str(BASE/"Photo_Interpretation_Data"/"Val"/"Images")
PI_TEST_IMG_DIR  = str(BASE/"Photo_Interpretation_Data"/"Test"/"Images")

# Máscaras GT semánticas (para métricas a nivel píxel y cobert./dens.)
GT_PI = str(BASE/"Photo_Interpretation_Data"/"Test"/"Annotations"/"Masks")
GT_FW = str(BASE/"Field_Work_Data"/"External_Val_Data"/"Annotations"/"Masks")

# Salidas específicas de PointRend (instancias)
OUT_PI = str(BASE/"PointRend_instance"/"output_PI")
OUT_FW = str(BASE/"PointRend_instance"/"output_FW")
PRED_PI = f"{OUT_PI}/Predictions"  # predicciones (θ variable)
PRED_FW = f"{OUT_FW}/Predictions"

for d in [OUT_PI, OUT_FW, PRED_PI, PRED_FW, f"{OUT_PI}/csv", f"{OUT_PI}/tables", f"{OUT_PI}/plots"]:
    os.makedirs(d, exist_ok=True)

# ----- GSD y bins de tamaño (mismos del paper) -----
GSD_FALLBACK = 0.13  # m/píxel
SIZE_BINS = [
    ("XS",  0.13,   1.72),
    ("S",   1.72,   3.62),
    ("M",   3.62,   9.08),
    ("L",   9.08,  20.82),
    ("XL", 20.82,  41.06),
    ("XXL",41.06, float("inf")),
]

def size_label(area_m2: float):
    for name, lo, hi in SIZE_BINS:
        if lo <= area_m2 < hi:
            return name
    return "XS"

# ----- Semillas reproducibles -----
SEED = 1337
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
#@title 2. Registro de datasets COCO (instancia) — PI (train/val/test) y FW (test)
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances

# Helpers para registro idempotente (evita choques al re-ejecutar celdas)
def ensure_registered(name, json_path, image_root):
    try:
        listed = DatasetCatalog.list()
    except Exception:
        listed = []
    if name not in listed:
        register_coco_instances(name, {}, json_path, image_root)

# PI — train/val/test
ensure_registered("pi_train_inst", PI_COCO_TRAIN_JSON, PI_TRAIN_IMG_DIR)
ensure_registered("pi_val_inst",   PI_COCO_VAL_JSON,   PI_VAL_IMG_DIR)
ensure_registered("pi_test_inst",  PI_COCO_TEST_JSON,  PI_TEST_IMG_DIR)

# FW — solo test
ensure_registered("fw_test_inst",  FW_COCO_TEST_JSON,  FW_IMG_DIR)

# Alias convenientes
PI_TRAIN = "pi_train_inst"
PI_VAL   = "pi_val_inst"
PI_TEST  = "pi_test_inst"
FW_TEST  = "fw_test_inst"

# (Opcional) Avisar si las clases difieren entre datasets (por ejemplo, 'Juniperus' vs 'juniperus')
def _safe_thing_classes(name):
    try:
        return list(MetadataCatalog.get(name).thing_classes)
    except Exception:
        return None

cls_train = _safe_thing_classes(PI_TRAIN)
cls_val   = _safe_thing_classes(PI_VAL)
cls_test  = _safe_thing_classes(PI_TEST)
cls_fw    = _safe_thing_classes(FW_TEST)

def _fmt(x):
    return x if x is not None else "None"

if any(x is None for x in [cls_train, cls_val, cls_test, cls_fw]):
    print("[WARN] No se pudieron leer thing_classes de algún dataset.")
else:
    if not (cls_train == cls_val == cls_test):
        print(f"[WARN] Diferencia de clases PI: train={cls_train}, val={cls_val}, test={cls_test}")
    if cls_fw != cls_train:
        print(f"[WARN] Clases en FW difieren de PI: fw={cls_fw} vs pi_train={cls_train}")

[WARN] No se pudieron leer thing_classes de algún dataset.


In [4]:
#@title 3. Configuración + (opcional) entrenamiento (PointRend Instancias, AUG compatible)
import os, torch
from detectron2.config import get_cfg
from detectron2.projects.point_rend import add_pointrend_config
from detectron2.engine import DefaultTrainer
from detectron2.data import build_detection_train_loader
from detectron2.data import DatasetMapper
from detectron2.data import transforms as T

cfg = get_cfg(); add_pointrend_config(cfg)

# YAML base (R50-FPN). Cambia a R101 si quieres más capacidad.
cfg.merge_from_file("/content/detectron2/projects/PointRend/configs/InstanceSegmentation/pointrend_rcnn_R_50_FPN_3x_coco.yaml")

# ---------------- Datasets ----------------
cfg.DATASETS.TRAIN = (PI_TRAIN,)
cfg.DATASETS.TEST  = (PI_VAL,)  # evaluación durante entrenamiento

# ---------------- Clases ------------------
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.POINT_HEAD.NUM_CLASSES = 1  # debe coincidir

# ---------------- Dispositivo/Batch -------
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.SOLVER.IMS_PER_BATCH = 2

# ---------------- Optimizador/Scheduler ---
cfg.SOLVER.MAX_ITER = 10000
cfg.SOLVER.BASE_LR  = 1e-3
cfg.SOLVER.STEPS    = [8000]
cfg.SOLVER.GAMMA    = 0.1
cfg.SOLVER.AMP.ENABLED = True
cfg.SOLVER.CLIP_GRADIENTS.ENABLED    = True
cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE = 1.0

# ---------------- RPN / ROI (más recall en pequeños) ------------
cfg.MODEL.RPN.PRE_NMS_TOPK_TRAIN  = 4000
cfg.MODEL.RPN.PRE_NMS_TOPK_TEST   = 4000
cfg.MODEL.RPN.POST_NMS_TOPK_TRAIN = 2000
cfg.MODEL.RPN.POST_NMS_TOPK_TEST  = 2000
cfg.MODEL.RPN.NMS_THRESH = 0.7

cfg.MODEL.ANCHOR_GENERATOR.SIZES         = [[8], [16], [32], [64], [128]]
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[1.0]]

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256
cfg.MODEL.ROI_HEADS.POSITIVE_FRACTION    = 0.5
cfg.MODEL.ROI_BOX_HEAD.CLS_AGNOSTIC_BBOX_REG = True
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST      = 0.5
cfg.MODEL.ROI_MASK_HEAD.LOSS_WEIGHT      = 2.0

# ---------------- PointRend reforzado ---------------------------
cfg.MODEL.POINT_HEAD.TRAIN_NUM_POINTS        = 4096
cfg.MODEL.POINT_HEAD.OVERSAMPLE_RATIO        = 3.0
cfg.MODEL.POINT_HEAD.IMPORTANCE_SAMPLE_RATIO = 0.9
cfg.MODEL.POINT_HEAD.NUM_FC_LAYERS           = 3
cfg.MODEL.POINT_HEAD.FC_DIM                  = 256
cfg.MODEL.POINT_HEAD.SUBDIVISION_STEPS       = 5
cfg.MODEL.POINT_HEAD.SUBDIVISION_NUM_POINTS  = 8192

# ---------------- Heads más finos (contorno/IoU@0.75) ----------
cfg.MODEL.ROI_MASK_HEAD.POOLER_RESOLUTION      = 28
cfg.MODEL.ROI_MASK_HEAD.CONV_DIM               = 256
cfg.MODEL.ROI_MASK_HEAD.NUM_CONV               = 6
cfg.MODEL.ROI_BOX_HEAD.POOLER_SAMPLING_RATIO   = 2
cfg.MODEL.ROI_MASK_HEAD.POOLER_SAMPLING_RATIO  = 2

# ---------------- Input sizes para inferencia ------------------
cfg.INPUT.MIN_SIZE_TRAIN = (800, 1024, 1200)
cfg.INPUT.MAX_SIZE_TRAIN = 1600
cfg.INPUT.MIN_SIZE_TEST  = 1024
cfg.INPUT.MAX_SIZE_TEST  = 1600
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.05

# ---------------- Salidas ----------------
cfg.OUTPUT_DIR = OUT_PI
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ---------------- AUGS compatibles ----------------
def build_train_augs(cfg):
    augs = []
    # Flips H/V
    augs.append(T.RandomFlip(horizontal=True,  vertical=False))
    augs.append(T.RandomFlip(horizontal=False, vertical=True))

    # Rotación más leve
    try:
        augs.append(T.RandomRotation(angle=[-10, 10], sample_style="range", expand=True))
    except TypeError:
        augs.append(T.RandomRotation([-10, 10]))

    # Resize multiescala estable
    augs.append(T.ResizeShortestEdge(
        short_edge_length=cfg.INPUT.MIN_SIZE_TRAIN,
        max_size=cfg.INPUT.MAX_SIZE_TRAIN,
        sample_style="choice"
    ))

    # Crop un poco menos agresivo (evita cortar contornos)
    augs.append(T.RandomCrop(crop_type="relative_range", crop_size=(0.9, 0.9)))

    # Fotometría muy leve (mantener textura y tono)
    augs.append(T.RandomBrightness(0.95, 1.05))
    augs.append(T.RandomContrast(0.95, 1.05))
    try:
        augs.append(T.RandomSaturation(0.95, 1.05))
    except AttributeError:
        pass

    # Sin blur en tu build
    return augs

class TrainerWithAug(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        mapper = DatasetMapper(
            cfg,
            is_train=True,
            augmentations=build_train_augs(cfg),
            image_format="BGR",
            use_instance_mask=True,
            recompute_boxes=True,
        )
        return build_detection_train_loader(cfg, mapper=mapper)

# ---------------- Entrenamiento opcional ----------------
DO_TRAIN = False
if DO_TRAIN:
    trainer = TrainerWithAug(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()

# Cargar pesos finales si existen (para inferencia/evaluación posteriores)
final_ckpt = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
if os.path.exists(final_ckpt):
    cfg.MODEL.WEIGHTS = final_ckpt

In [5]:
#@title 4. Predictor (instancias) + utilidades
from detectron2.engine import DefaultPredictor
from pycocotools.coco import COCO
from scipy import ndimage

predictor = DefaultPredictor(cfg)

# --- utilidades de georreferenciación / GSD ---
def image_gsd_m(path, fallback=GSD_FALLBACK):
    try:
        with rasterio.open(path) as src:
            tr = src.transform
            if src.crs and src.crs.is_projected:
                px_m = abs(tr.a); py_m = abs(tr.e)
            else:
                # aproximación grosa para grados geográficos
                mx = 111320.0; my = 110540.0
                px_m = mx * abs(tr.a); py_m = my * abs(tr.e)
            if px_m > 0 and py_m > 0:
                return float((px_m + py_m) / 2.0)
    except Exception:
        pass
    return float(fallback)

# --- predicción de instancias (sin TTA) ---
def predict_instances(img_bgr):
    out = predictor(img_bgr)
    inst = out["instances"].to("cpu")
    masks = []
    scores = []
    if inst.has("pred_masks"):
        pm = inst.pred_masks.numpy()  # [N,H,W] bool
        sc = inst.scores.numpy().tolist()
        for m, s in zip(pm, sc):
            masks.append(m.astype(bool))
            scores.append(float(s))
    return masks, scores

In [6]:
#@title 5. Evaluación — IoU / S-IoU, curvas F1 vs θ_score, tamaños, pixel-metrics
import os, cv2, json, math, numpy as np, pandas as pd
from pycocotools.coco import COCO
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt

# -------- Helpers COCO --------
def ann_masks_for_img(coco: COCO, img_id: int, H: int, W: int):
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)
    masks = []
    for a in anns:
        m = coco.annToMask(a).astype(bool)
        if m.shape != (H, W):
            m = cv2.resize(
                m.astype(np.uint8), (W, H),
                interpolation=cv2.INTER_NEAREST
            ).astype(bool)
        masks.append(m)
    return masks

# -------- IoU / S-IoU --------
def iou_matrix(pred_masks, gt_masks):
    if len(pred_masks)==0 or len(gt_masks)==0:
        return np.zeros((len(pred_masks), len(gt_masks)), dtype=float)
    P, G = len(pred_masks), len(gt_masks)
    M = np.zeros((P, G), dtype=float)
    for i, p in enumerate(pred_masks):
        p_area = p.sum()
        for j, g in enumerate(gt_masks):
            inter = np.logical_and(p, g).sum()
            union = p_area + g.sum() - inter
            M[i, j] = float(inter / max(1, union))
    return M

# S-IoU según tu definición (pred vs unión de GT solapadas y viceversa)
def siou_pred(p_mask: np.ndarray, gt_masks):
    matches = [g for g in gt_masks if np.any(p_mask & g)]
    if not matches:
        return 0.0
    union_gt = np.any(np.stack(matches, axis=0), axis=0)
    inter = np.logical_and(p_mask, union_gt).sum()
    den = union_gt.sum()
    return float(inter / den) if den > 0 else 0.0

def siou_label(l_mask: np.ndarray, pred_masks):
    matches = [p for p in pred_masks if np.any(l_mask & p)]
    if not matches:
        return 0.0
    union_pr = np.any(np.stack(matches, axis=0), axis=0)
    inter = np.logical_and(l_mask, union_pr).sum()
    den = l_mask.sum()
    return float(inter / den) if den > 0 else 0.0

# Matching húngaro para IoU@thr
def eval_iou_at_threshold_hungarian(pred_masks, gt_masks, iou_thr=0.5):
    iou = iou_matrix(pred_masks, gt_masks)
    if iou.size == 0:
        return dict(tp=0, fp=len(pred_masks), fn=len(gt_masks))
    cost = 1.0 - iou
    ri, cj = linear_sum_assignment(cost)
    matched_pred = set(); matched_gt = set()
    for i, j in zip(ri, cj):
        if iou[i, j] >= iou_thr:
            matched_pred.add(i); matched_gt.add(j)
    tp = len(matched_pred)
    fp = len(pred_masks) - tp
    fn = len(gt_masks) - tp
    return dict(tp=tp, fp=fp, fn=fn)

# S-IoU@thr con barrido por score (ordenación por score desc)
def eval_siou_at_threshold(pred_masks, pred_scores, gt_masks, siou_thr=0.5):
    order = np.argsort(-np.asarray(pred_scores)) if len(pred_scores)>0 else np.arange(len(pred_masks))
    tp = fp = 0
    for i in order:
        s = siou_pred(pred_masks[i], gt_masks)
        if s >= siou_thr:
            tp += 1
        else:
            fp += 1
    fn = 0
    for g in gt_masks:
        s = siou_label(g, pred_masks)
        if s < siou_thr:
            fn += 1
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1)

# -------- Tamaños / pixel-metrics --------
def size_label(area_m2: float):
    for name, lo, hi in SIZE_BINS:
        if lo <= area_m2 < hi:
            return name
    return "XS"

def pixel_metrics(pred_binary: np.ndarray, gt_binary: np.ndarray):
    y_pred = pred_binary.astype(np.uint8).ravel()
    y_true = gt_binary.astype(np.uint8).ravel()
    tp = int(np.sum((y_true==1) & (y_pred==1)))
    tn = int(np.sum((y_true==0) & (y_pred==0)))
    fp = int(np.sum((y_true==0) & (y_pred==1)))
    fn = int(np.sum((y_true==1) & (y_pred==0)))
    total = tp + tn + fp + fn
    acc = (tp + tn) / total if total > 0 else 0.0
    iou_fg = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    iou_bg = tn / (tn + fp + fn) if (tn + fp + fn) > 0 else 0.0
    miou = 0.5 * (iou_fg + iou_bg)
    w0 = (tn + fp) / total if total > 0 else 0.0
    w1 = (tp + fn) / total if total > 0 else 0.0
    fwiou = w0 * iou_bg + w1 * iou_fg
    return dict(pACC=acc, mIoU=miou, fwIoU=fwiou)

# -------- Conversión instancias -> binario y evaluación completa --------
def evaluate_instances_vs_coco(
    coco_json,
    img_dir,
    predictor_func,
    write_pred_bin_dir=None,
    model_tag="PointRend-Inst",
    theta_grid=np.linspace(0.05, 0.95, 19),
    iou_thrs=(0.5, 0.75),
    siou_thrs=(0.5, 0.75)
):
    coco = COCO(coco_json)
    images = coco.loadImgs(coco.getImgIds())

    # estos contadores 'res_counts' se dejan, pero no se agregan; usamos curves_counts
    res_counts = {("IoU",t): dict(tp=0,fp=0,fn=0) for t in iou_thrs}
    res_counts.update({("S-IoU",t): dict(tp=0,fp=0,fn=0) for t in siou_thrs})

    curves_counts = {("IoU",t): [dict(tp=0,fp=0,fn=0) for _ in theta_grid] for t in iou_thrs}
    curves_counts.update({("S-IoU",t): [dict(tp=0,fp=0,fn=0) for _ in theta_grid] for t in siou_thrs})

    sizewise = {name: {("IoU",0.5):dict(tp=0,fp=0,fn=0),
                       ("S-IoU",0.5):dict(tp=0,fp=0,fn=0)}
                for name,_,_ in SIZE_BINS}
    sizewise["All"] = {("IoU",0.5):dict(tp=0,fp=0,fn=0),
                       ("S-IoU",0.5):dict(tp=0,fp=0,fn=0)}

    pix_agg = dict(pACC=[], mIoU=[], fwIoU=[])

    for im in tqdm(images, desc=f"Eval {model_tag}"):
        fpath = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fpath)
        if img is None:
            continue
        H, W = img.shape[:2]
        gsd = image_gsd_m(fpath, fallback=GSD_FALLBACK)

        gt_masks = ann_masks_for_img(coco, im["id"], H, W)

        # predicción de instancias
        pred_masks_all, pred_scores_all = predictor_func(img)

        # binario por unión de instancias (para pix-metrics y escritura opcional)
        pred_bin = np.zeros((H, W), dtype=np.uint8)
        for m, s in zip(pred_masks_all, pred_scores_all):
            pred_bin[m] = 1

        if write_pred_bin_dir:
            os.makedirs(write_pred_bin_dir, exist_ok=True)
            with rasterio.open(fpath) as src:
                meta = src.meta.copy()
                meta.update(count=1, dtype=rasterio.uint8, nodata=0)
                out_path = os.path.join(
                    write_pred_bin_dir,
                    os.path.basename(fpath).replace("Img_","Mask_")
                )
                with rasterio.open(out_path, "w", **meta) as dst:
                    dst.write(pred_bin, 1)

        gt_bin = np.any(np.stack(gt_masks,0),0) if len(gt_masks)>0 else np.zeros((H,W), bool)
        pm = pixel_metrics(pred_bin.astype(bool), gt_bin.astype(bool))
        for k,v in pm.items():
            pix_agg[k].append(v)

        # Barrido por theta (score) -> counts por métrica/umbral
        for tidx, theta in enumerate(theta_grid):
            sel  = [i for i,s in enumerate(pred_scores_all) if s >= float(theta)]
            pm_t = [pred_masks_all[i] for i in sel]
            ps_t = [pred_scores_all[i] for i in sel]

            for t in iou_thrs:
                m = eval_iou_at_threshold_hungarian(pm_t, gt_masks, iou_thr=t)
                for k in ("tp","fp","fn"):
                    curves_counts[("IoU",t)][tidx][k] += m[k]

            for t in siou_thrs:
                m = eval_siou_at_threshold(pm_t, ps_t, gt_masks, siou_thr=t)
                for k in ("tp","fp","fn"):
                    curves_counts[("S-IoU",t)][tidx][k] += m[k]

        # Size-wise @0.5 (IoU y S-IoU), sin filtrar por score
        iou_mat  = iou_matrix(pred_masks_all, gt_masks)
        assigned = set()
        gt_areas = [float(g.sum())*(gsd**2) for g in gt_masks] if len(gt_masks)>0 else []

        if iou_mat.size>0:
            order = np.argsort(-iou_mat.max(axis=1))
            for i in order:
                if iou_mat.shape[1]==0:
                    break
                j = int(np.argmax(iou_mat[i]))
                if iou_mat[i,j] >= 0.5 and j not in assigned:
                    assigned.add(j)
                    sname = size_label(gt_areas[j])
                    sizewise[sname][("IoU",0.5)]["tp"] += 1
                    sizewise["All"][("IoU",0.5)]["tp"] += 1

        for j in range(len(gt_masks)):
            if j not in assigned:
                sname = size_label(gt_areas[j] if j < len(gt_areas) else 0.0)
                sizewise[sname][("IoU",0.5)]["fn"] += 1
                sizewise["All"][("IoU",0.5)]["fn"] += 1

        for i in range(len(pred_masks_all)):
            if iou_mat.shape[1]==0 or iou_mat[i].max() < 0.5:
                area_m2 = float(pred_masks_all[i].sum())*(gsd**2)
                sname   = size_label(area_m2)
                sizewise[sname][("IoU",0.5)]["fp"] += 1
                sizewise["All"][("IoU",0.5)]["fp"] += 1

        # S-IoU size-wise @0.5
        for p in pred_masks_all:
            overlaps = [g for g in gt_masks if np.any(p & g)]
            if not overlaps:
                continue
            union_gt = np.any(np.stack(overlaps, axis=0), axis=0)
            union_area_m2 = float(union_gt.sum())*(gsd**2)
            bname = size_label(union_area_m2)
            s_pred = siou_pred(p, gt_masks)
            if s_pred >= 0.5:
                sizewise[bname][("S-IoU",0.5)]["tp"] += 1
                sizewise["All"][("S-IoU",0.5)]["tp"] += 1
            else:
                sizewise[bname][("S-IoU",0.5)]["fp"] += 1
                sizewise["All"][("S-IoU",0.5)]["fp"] += 1

        for g in gt_masks:
            s_lab = siou_label(g, pred_masks_all)
            if s_lab < 0.5:
                g_area_m2 = float(g.sum())*(gsd**2)
                bname = size_label(g_area_m2)
                sizewise[bname][("S-IoU",0.5)]["fn"] += 1
                sizewise["All"][("S-IoU",0.5)]["fn"] += 1

    # Aggregate helpers
    def counts_to_metrics(d):
        tp,fp,fn = d["tp"], d["fp"], d["fn"]
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        return dict(tp=tp,fp=fp,fn=fn,precision=prec,recall=rec,f1=f1)

    # Curvas F1
    def agg_curves_to_f1(curves_counts):
        out = {}
        for key, lst in curves_counts.items():
            f1s = []
            for d in lst:
                f1s.append(counts_to_metrics(d)["f1"])
            out[key] = f1s
        return out

    metrics     = {k: counts_to_metrics(v) for k,v in res_counts.items()}
    f1_curves   = agg_curves_to_f1(curves_counts)
    pix_summary = {k: float(np.mean(v)) if len(v)>0 else 0.0 for k,v in pix_agg.items()}
    size_metrics = {
        sname:{key:counts_to_metrics(cnt) for key,cnt in sub.items()}
        for sname,sub in sizewise.items()
    }

    # Devolvemos curves_counts para poder leer TP/FP/FN al θ que elijas
    res = dict(
        metrics=metrics,              # no usado para tabla final
        f1_curves=f1_curves,          # usado para θ*
        thr_grid=list(map(float, theta_grid)),
        pixel_metrics=pix_summary,
        size_metrics=size_metrics,
        curves_counts=curves_counts   # <-- para leer counts
    )
    return res


# -------- Curvas estilo paper (4 líneas) --------
def _res_to_curves_df(res):
    grid = np.array(res["thr_grid"], dtype=float)
    return pd.DataFrame({
        "theta": grid,
        "F1_IoU_0p5":  np.array(res["f1_curves"][("IoU", 0.5)], dtype=float),
        "F1_IoU_0p75": np.array(res["f1_curves"][("IoU", 0.75)], dtype=float),
        "F1_SIoU_0p5": np.array(res["f1_curves"][("S-IoU",0.5)], dtype=float),
        "F1_SIoU_0p75":np.array(res["f1_curves"][("S-IoU",0.75)], dtype=float),
    })

COLS_STD = ["F1_IoU_0.5","F1_IoU_0.75","F1_SIoU_0.5","F1_SIoU_0.75"]

def _percent_and_rename(df):
    df_out = df.rename(columns={
        "F1_IoU_0p5":"F1_IoU_0.5",
        "F1_IoU_0p75":"F1_IoU_0.75",
        "F1_SIoU_0p5":"F1_SIoU_0.5",
        "F1_SIoU_0p75":"F1_SIoU_0.75",
    }).copy()
    df_out[COLS_STD] = df_out[COLS_STD] * 100.0
    return df_out

def _plot_four_curves(df, split_name, theta_star=None, out_dir=OUT_PI):
    plt.rcParams.update({
        "font.size":11,"axes.titlesize":12,"axes.labelsize":11,
        "legend.fontsize":10,"figure.dpi":150
    })
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(df["theta"], 100*df["F1_IoU_0p5"],  marker="o", label="IoU @ 0.5")
    ax.plot(df["theta"], 100*df["F1_IoU_0p75"], marker="s", label="IoU @ 0.75")
    ax.plot(df["theta"], 100*df["F1_SIoU_0p5"],  marker="^", label="S-IoU @ 0.5")
    ax.plot(df["theta"], 100*df["F1_SIoU_0p75"], marker="D", label="S-IoU @ 0.75")
    if theta_star is not None:
        ax.axvline(float(theta_star), linestyle="--", linewidth=1)
    ax.set_xlabel("θ_score"); ax.set_ylabel("F1-score (%)")
    ax.set_title(f"{split_name} — F1 vs θ_score — PointRend Inst")
    ax.grid(True, alpha=0.3); ax.legend(loc="best", frameon=False)
    os.makedirs(f"{out_dir}/plots", exist_ok=True)
    out_path = os.path.join(out_dir, "plots",
                            f"F1_vs_theta_{split_name}_4curves_PointRendInst.png")
    plt.savefig(out_path, bbox_inches="tight"); plt.close()
    return out_path

def _theta_star_from(res, key=("S-IoU",0.5)):
    grid = np.asarray(res["thr_grid"], dtype=float)
    f1   = np.asarray(res["f1_curves"][key], dtype=float)
    i    = int(np.argmax(f1))
    return float(grid[i]), float(f1[i])

# === NUEVO: leer counts en el θ más cercano para un key dado ===
def _counts_at_theta(res, theta, key):
    """
    key = ("IoU", 0.5) | ("IoU", 0.75) | ("S-IoU", 0.5) | ("S-IoU", 0.75)
    Devuelve dict(tp, fp, fn) en el índice de grid más cercano a 'theta'.
    """
    grid = np.asarray(res["thr_grid"], dtype=float)
    idx  = int(np.argmin(np.abs(grid - float(theta))))
    d    = res["curves_counts"][key][idx]
    return dict(tp=d["tp"], fp=d["fp"], fn=d["fn"])

# === REHECHO: construir tabla global usando los counts @ θ elegido ===
def to_df_global(res_or_metrics, data_tag, theta):
    """
    Acepta:
    - Salida de evaluate_instances_vs_coco (tiene 'thr_grid' y 'curves_counts')
    - Salida de evaluate_instances_at_theta (tiene 'metrics' con tp/fp/fn ya a θ)
    - O un dict {key -> {tp,fp,fn}} directamente.
    """
    def counts_to_metrics_local(d):
        tp, fp, fn = d["tp"], d["fp"], d["fn"]
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        return dict(tp=tp, fp=fp, fn=fn,
                    precision=prec, recall=rec, f1=f1)

    # Decide cómo obtener los counts según el tipo de entrada
    if isinstance(res_or_metrics, dict) and "curves_counts" in res_or_metrics and "thr_grid" in res_or_metrics:
        # Formato evaluate_instances_vs_coco
        def get_counts(key):
            return _counts_at_theta(res_or_metrics, theta, key)
    elif isinstance(res_or_metrics, dict) and "metrics" in res_or_metrics:
        # Formato evaluate_instances_at_theta
        def get_counts(key):
            m = res_or_metrics["metrics"][key]
            return dict(tp=m["tp"], fp=m["fp"], fn=m["fn"])
    else:
        # Dict directo: {key -> {tp,fp,fn}}
        def get_counts(key):
            return res_or_metrics[key]

    rows = []
    for key in [("IoU",0.5), ("IoU",0.75), ("S-IoU",0.5), ("S-IoU",0.75)]:
        cnt = get_counts(key)
        m   = counts_to_metrics_local(cnt)
        rows.append(dict(
            Data=f"{data_tag} (θ={theta:.2f})",
            Metric=key[0],
            Thr=key[1],
            TP=m["tp"], FP=m["fp"], FN=m["fn"],
            Precision=100*m["precision"],
            Recall=100*m["recall"],
            F1_score=100*m["f1"]
        ))
    return pd.DataFrame(rows)

In [7]:
#@title 6. Curvas PI/FW (4 líneas) + selección de θ* (S-IoU@0.5) y guardados
# Evaluación sobre PI (test) y FW (test). Genera CSVs y PNGs.

print("PI (curvas vs θ_score)…")
res_pi = evaluate_instances_vs_coco(
    PI_COCO_TEST_JSON, PI_TEST_IMG_DIR, predict_instances,
    write_pred_bin_dir=PRED_PI,
    model_tag="PointRend-Inst-PI",
    theta_grid=np.linspace(0.05, 0.95, 19)
)
df_curves_pi = _res_to_curves_df(res_pi)
theta_pi, f1_pi = _theta_star_from(res_pi, ("S-IoU",0.5))
os.makedirs(OUT_PI, exist_ok=True)
df_curves_pi_out = _percent_and_rename(df_curves_pi)
df_curves_pi_out.to_csv(
    os.path.join(OUT_PI, "curves_F1_4lines_PI_PointRendInst.csv"),
    index=False
)
png_pi = _plot_four_curves(df_curves_pi, "PI", theta_star=theta_pi, out_dir=OUT_PI)
print(f"PI: θ* (S-IoU@0.5) = {theta_pi:.2f} | F1={100*f1_pi:.2f}%")
print("Curvas PI guardadas en:", png_pi)

print("FW (curvas vs θ_score)…")
res_fw = evaluate_instances_vs_coco(
    FW_COCO_TEST_JSON, FW_IMG_DIR, predict_instances,
    write_pred_bin_dir=PRED_FW,
    model_tag="PointRend-Inst-FW",
    theta_grid=np.linspace(0.05, 0.95, 19)
)
df_curves_fw = _res_to_curves_df(res_fw)
theta_fw, f1_fw = _theta_star_from(res_fw, ("S-IoU",0.5))
df_curves_fw_out = _percent_and_rename(df_curves_fw)
df_curves_fw_out.to_csv(
    os.path.join(OUT_PI, "curves_F1_4lines_FW_PointRendInst.csv"),
    index=False
)
png_fw = _plot_four_curves(df_curves_fw, "FW", theta_star=theta_fw, out_dir=OUT_PI)
print(f"FW: θ* (S-IoU@0.5) = {theta_fw:.2f} | F1={100*f1_fw:.2f}%")
print("Curvas FW guardadas en:", png_fw)

# Tablas globales a θ* usando los counts reales del grid
df_pi  = to_df_global(res_pi, "PI-test", theta_pi)
df_fw  = to_df_global(res_fw, "FW-test", theta_fw)
df_all = pd.concat([df_pi, df_fw], ignore_index=True)

from IPython.display import display
try:
    display(df_all)
except Exception:
    pass

csv_dir = f"{OUT_PI}/csv"
tab_dir = f"{OUT_PI}/tables"
os.makedirs(csv_dir, exist_ok=True); os.makedirs(tab_dir, exist_ok=True)
df_all.to_csv(f"{csv_dir}/global_metrics_PointRendInst_theta_star.csv", index=False)
with open(f"{tab_dir}/table_global_PointRendInst_theta_star.tex","w") as f:
    f.write(df_all.to_latex(index=False, float_format='%.4f'))

PI (curvas vs θ_score)…
loading annotations into memory...
Done (t=2.35s)
creating index...
index created!


Eval PointRend-Inst-PI:   0%|          | 0/75 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W1202 16:22:21.789000 876 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Eval PointRend-Inst-PI: 100%|██████████| 75/75 [13:35<00:00, 10.87s/it]


PI: θ* (S-IoU@0.5) = 0.90 | F1=85.16%
Curvas PI guardadas en: /content/drive/MyDrive/juniper_mapper/JuniperMapper/PointRend_instance/output_PI/plots/F1_vs_theta_PI_4curves_PointRendInst.png
FW (curvas vs θ_score)…
loading annotations into memory...
Done (t=10.74s)
creating index...
index created!


Eval PointRend-Inst-FW: 100%|██████████| 124/124 [28:55<00:00, 14.00s/it]


FW: θ* (S-IoU@0.5) = 0.75 | F1=74.17%
Curvas FW guardadas en: /content/drive/MyDrive/juniper_mapper/JuniperMapper/PointRend_instance/output_PI/plots/F1_vs_theta_FW_4curves_PointRendInst.png


,Data,Metric,Thr,TP,FP,FN,Precision,Recall,F1_score
0,PI-test (θ=0.90),IoU,0.50,606,133,84,82.002706,87.826087,84.814556
1,PI-test (θ=0.90),IoU,0.75,499,240,191,67.523681,72.318841,69.839048
2,PI-test (θ=0.90),S-IoU,0.50,588,151,54,79.566982,91.588785,85.155684
3,PI-test (θ=0.90),S-IoU,0.75,508,231,103,68.741543,83.142390,75.259259
4,FW-test (θ=0.75),IoU,0.50,1275,518,496,71.109872,71.993224,71.548822
5,FW-test (θ=0.75),IoU,0.75,803,990,968,44.785276,45.341615,45.061728
6,FW-test (θ=0.75),S-IoU,0.50,1288,505,392,71.834914,76.666667,74.172185
7,FW-test (θ=0.75),S-IoU,0.75,984,809,560,54.880089,63.730570,58.975127


In [8]:
#@title 7. Métricas a θ* (PI y FW) + guardado de máscaras a θ*
import os, numpy as np, pandas as pd, rasterio
from pycocotools.coco import COCO
from tqdm import tqdm

# Wrapper: evalúa a un θ concreto (filtrado por score >= θ) y opcionalmente guarda máscaras
def evaluate_instances_at_theta(
    coco_json, img_dir, predictor_func, theta_star,
    write_pred_bin_dir=None,
    model_tag="PointRend-Inst@theta",
    iou_thrs=(0.5,0.75), siou_thrs=(0.5,0.75)
):
    coco = COCO(coco_json)
    images = coco.loadImgs(coco.getImgIds())

    res_counts = {("IoU",t): dict(tp=0,fp=0,fn=0) for t in iou_thrs}
    res_counts.update({("S-IoU",t): dict(tp=0,fp=0,fn=0) for t in siou_thrs})

    sizewise = {name: {("IoU",0.5):dict(tp=0,fp=0,fn=0), ("S-IoU",0.5):dict(tp=0,fp=0,fn=0)} for name,_,_ in SIZE_BINS}
    sizewise["All"] = {("IoU",0.5):dict(tp=0,fp=0,fn=0), ("S-IoU",0.5):dict(tp=0,fp=0,fn=0)}

    pix_agg = dict(pACC=[], mIoU=[], fwIoU=[])

    for im in tqdm(images, desc=f"Eval {model_tag} θ={theta_star:.2f}"):
        fpath = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fpath)
        if img is None:
            continue
        H, W = img.shape[:2]
        gsd = image_gsd_m(fpath, fallback=GSD_FALLBACK)
        gt_masks = ann_masks_for_img(coco, im["id"], H, W)

        pred_masks_all, pred_scores_all = predict_instances(img)

        # filtra por score >= theta
        idx = [i for i,s in enumerate(pred_scores_all) if s >= float(theta_star)]
        pred_masks = [pred_masks_all[i] for i in idx]
        pred_scores = [pred_scores_all[i] for i in idx]

        # binario por unión (para pix-metrics y escritura)
        pred_bin = np.zeros((H, W), dtype=np.uint8)
        for m in pred_masks:
            pred_bin[m] = 1

        if write_pred_bin_dir:
            os.makedirs(write_pred_bin_dir, exist_ok=True)
            with rasterio.open(fpath) as src:
                meta = src.meta.copy(); meta.update(count=1, dtype=rasterio.uint8, nodata=0)
                out_path = os.path.join(write_pred_bin_dir, os.path.basename(fpath).replace("Img_","Mask_"))
                with rasterio.open(out_path, "w", **meta) as dst:
                    dst.write(pred_bin, 1)

        # métricas pixelares por imagen (usando helpers ya definidos)
        gt_bin = np.any(np.stack(gt_masks,0),0) if len(gt_masks)>0 else np.zeros((H,W), bool)
        pm = pixel_metrics(pred_bin.astype(bool), gt_bin.astype(bool))
        for k,v in pm.items():
            pix_agg[k].append(v)

        # Global @ IoU y S-IoU
        for t in iou_thrs:
            m = eval_iou_at_threshold_hungarian(pred_masks, gt_masks, iou_thr=t)
            for k in ("tp","fp","fn"): res_counts[("IoU",t)][k] += m[k]
        for t in siou_thrs:
            m = eval_siou_at_threshold(pred_masks, pred_scores, gt_masks, siou_thr=t)
            for k in ("tp","fp","fn"): res_counts[("S-IoU",t)][k] += m[k]

        # Size-wise @0.5 (IoU)
        iou_mat = iou_matrix(pred_masks, gt_masks)
        assigned=set(); gt_areas=[float(g.sum())*(gsd**2) for g in gt_masks] if len(gt_masks)>0 else []
        if iou_mat.size>0:
            order = np.argsort(-iou_mat.max(axis=1))
            for i in order:
                if iou_mat.shape[1]==0: break
                j = int(np.argmax(iou_mat[i]))
                if iou_mat[i,j] >= 0.5 and j not in assigned:
                    assigned.add(j)
                    sname=size_label(gt_areas[j])
                    sizewise[sname][("IoU",0.5)]["tp"] += 1
                    sizewise["All"][("IoU",0.5)]["tp"] += 1
        for j in range(len(gt_masks)):
            if j not in assigned:
                sname=size_label(gt_areas[j] if j < len(gt_areas) else 0.0)
                sizewise[sname][("IoU",0.5)]["fn"] += 1
                sizewise["All"][("IoU",0.5)]["fn"] += 1
        for i in range(len(pred_masks)):
            if iou_mat.shape[1]==0 or iou_mat[i].max() < 0.5:
                area_m2 = float(pred_masks[i].sum())*(gsd**2)
                sname=size_label(area_m2)
                sizewise[sname][("IoU",0.5)]["fp"] += 1
                sizewise["All"][("IoU",0.5)]["fp"] += 1

        # Size-wise @0.5 (S-IoU)
        for p in pred_masks:
            overlaps = [g for g in gt_masks if np.any(p & g)]
            if not overlaps: continue
            union_gt = np.any(np.stack(overlaps, axis=0), axis=0)
            union_area_m2 = float(union_gt.sum())*(gsd**2)
            bname = size_label(union_area_m2)
            s_pred = siou_pred(p, gt_masks)
            if s_pred >= 0.5:
                sizewise[bname][("S-IoU",0.5)]["tp"] += 1
                sizewise["All"][("S-IoU",0.5)]["tp"] += 1
            else:
                sizewise[bname][("S-IoU",0.5)]["fp"] += 1
                sizewise["All"][("S-IoU",0.5)]["fp"] += 1
        for g in gt_masks:
            s_lab = siou_label(g, pred_masks)
            if s_lab < 0.5:
                g_area_m2 = float(g.sum())*(gsd**2)
                bname = size_label(g_area_m2)
                sizewise[bname][("S-IoU",0.5)]["fn"] += 1
                sizewise["All"][("S-IoU",0.5)]["fn"] += 1

    # Aggregate
    def counts_to_metrics(d):
        tp,fp,fn = d["tp"], d["fp"], d["fn"]
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        return dict(tp=tp,fp=fp,fn=fn,precision=prec,recall=rec,f1=f1)

    metrics = {k: counts_to_metrics(v) for k,v in res_counts.items()}
    pix_summary = {k: float(np.mean(v)) if len(v)>0 else 0.0 for k,v in pix_agg.items()}
    size_metrics = {sname:{key:counts_to_metrics(cnt) for key,cnt in sub.items()} for sname,sub in sizewise.items()}

    return dict(metrics=metrics, pixel_metrics=pix_summary, size_metrics=size_metrics)

print("Evaluando a θ* y guardando máscaras…")
PRED_PI_TH = f"{OUT_PI}/Predictions_theta_star"
PRED_FW_TH = f"{OUT_FW}/Predictions_theta_star"

res_pi_star = evaluate_instances_at_theta(
    PI_COCO_TEST_JSON, PI_TEST_IMG_DIR, predict_instances, theta_pi,
    write_pred_bin_dir=PRED_PI_TH, model_tag="PointRend-Inst@PI"
)
res_fw_star = evaluate_instances_at_theta(
    FW_COCO_TEST_JSON, FW_IMG_DIR, predict_instances, theta_fw,
    write_pred_bin_dir=PRED_FW_TH, model_tag="PointRend-Inst@FW"
)

# ---- Tablas globales (IoU/S-IoU @ {0.5, 0.75}) a θ* (sin usar to_df_global) ----
rows = []
for tag, res, theta in [("PI-test", res_pi_star, theta_pi), ("FW-test", res_fw_star, theta_fw)]:
    for (metric,thr), d in res["metrics"].items():
        rows.append(dict(
            Data=f"{tag} (θ={theta:.2f})",
            Metric=metric, Thr=thr,
            TP=d["tp"], FP=d["fp"], FN=d["fn"],
            Precision=100*d["precision"], Recall=100*d["recall"], F1_score=100*d["f1"]
        ))
df_all = pd.DataFrame(rows)

# ---- Tablas por tamaños (IoU @0.5 y S-IoU @0.5) a θ* ----
def _f1(p, r):
    return (2*p*r/max(1e-9, (p+r))) if (p+r)>0 else 0.0

def _sizewise_df(res, tag):
    out = []
    for s in [b[0] for b in SIZE_BINS] + ["All"]:
        ri = res["size_metrics"][s][("IoU",0.5)]
        rs = res["size_metrics"][s][("S-IoU",0.5)]
        out.append({
            "Data": tag, "Size": s,
            "IoU_P": 100*ri["precision"], "IoU_R": 100*ri["recall"], "IoU_F1": 100*_f1(ri["precision"], ri["recall"]),
            "S-IoU_P": 100*rs["precision"], "S-IoU_R": 100*rs["recall"], "S-IoU_F1": 100*_f1(rs["precision"], rs["recall"]),
        })
    return pd.DataFrame(out)

df_pi_sz = _sizewise_df(res_pi_star, "PI-test")
df_fw_sz = _sizewise_df(res_fw_star, "FW-test")

# ---- Mostrar (si hay entorno interactivo) ----
from IPython.display import display
try:
    display(df_all); display(df_pi_sz); display(df_fw_sz)
except Exception:
    pass

# ---- Guardados ----
os.makedirs(f"{OUT_PI}/csv", exist_ok=True)
os.makedirs(f"{OUT_PI}/tables", exist_ok=True)

df_all.to_csv(f"{OUT_PI}/csv/global_metrics_PointRendInst_theta_star.csv", index=False)
with open(f"{OUT_PI}/tables/table_global_PointRendInst_theta_star.tex","w") as f:
    f.write(df_all.to_latex(index=False, float_format='%.4f'))

df_pi_sz.to_csv(f"{OUT_PI}/csv/sizewise_PI_PointRendInst_theta_star.csv", index=False)
df_fw_sz.to_csv(f"{OUT_PI}/csv/sizewise_FW_PointRendInst_theta_star.csv", index=False)
with open(f"{OUT_PI}/tables/table_sizewise_PI_PointRendInst_theta_star.tex","w") as f:
    f.write(df_pi_sz.to_latex(index=False, float_format='%.2f'))
with open(f"{OUT_PI}/tables/table_sizewise_FW_PointRendInst_theta_star.tex","w") as f:
    f.write(df_fw_sz.to_latex(index=False, float_format='%.2f'))

Evaluando a θ* y guardando máscaras…
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


Eval PointRend-Inst@PI θ=0.90: 100%|██████████| 75/75 [01:41<00:00,  1.35s/it]


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


Eval PointRend-Inst@FW θ=0.75: 100%|██████████| 124/124 [03:25<00:00,  1.66s/it]


,Data,Metric,Thr,TP,FP,FN,Precision,Recall,F1_score
0,PI-test (θ=0.90),IoU,0.50,606,133,84,82.002706,87.826087,84.814556
1,PI-test (θ=0.90),IoU,0.75,499,240,191,67.523681,72.318841,69.839048
2,PI-test (θ=0.90),S-IoU,0.50,588,151,54,79.566982,91.588785,85.155684
3,PI-test (θ=0.90),S-IoU,0.75,508,231,103,68.741543,83.142390,75.259259
4,FW-test (θ=0.75),IoU,0.50,1275,518,496,71.109872,71.993224,71.548822
5,FW-test (θ=0.75),IoU,0.75,803,990,968,44.785276,45.341615,45.061728
6,FW-test (θ=0.75),S-IoU,0.50,1288,505,392,71.834914,76.666667,74.172185
7,FW-test (θ=0.75),S-IoU,0.75,984,809,560,54.880089,63.730570,58.975127


,Data,Size,IoU_P,IoU_R,IoU_F1,S-IoU_P,S-IoU_R,S-IoU_F1
0,PI-test,XS,20.000000,53.846154,29.166667,100.000000,58.333333,73.684211
1,PI-test,S,76.744186,82.500000,79.518072,100.000000,88.157895,93.706294
2,PI-test,M,79.518072,87.417219,83.280757,99.193548,87.857143,93.181818
3,PI-test,L,87.894737,88.359788,88.126649,90.163934,92.178771,91.160221
4,PI-test,XL,91.240876,93.283582,92.250923,89.516129,96.521739,92.887029
5,PI-test,XXL,90.082645,88.617886,89.344262,67.647059,95.833333,79.310345
6,PI-test,All,82.448980,87.826087,85.052632,87.111111,91.588785,89.293850


,Data,Size,IoU_P,IoU_R,IoU_F1,S-IoU_P,S-IoU_R,S-IoU_F1
0,FW-test,XS,54.337900,49.377593,51.739130,98.400000,53.246753,69.101124
1,FW-test,S,68.560606,62.413793,65.342960,97.905759,67.753623,80.085653
2,FW-test,M,72.160356,72.808989,72.483221,89.714286,76.960784,82.849604
3,FW-test,L,75.682382,84.254144,79.738562,82.222222,86.297376,84.210526
4,FW-test,XL,83.935743,83.935743,83.935743,65.517241,89.316239,75.587703
5,FW-test,XXL,84.567901,74.456522,79.190751,46.491228,84.574468,60.000000
6,FW-test,All,73.024055,71.993224,72.504976,76.348548,76.666667,76.507277


In [ ]:
#@title 8. Métricas pixelares a θ* (mIoU, pixAcc, fwIoU)

def read_mask(path):
    with rasterio.open(path) as src:
        return src.read(1)

def confusion_from_masks(gt, pr, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    gt = gt.flatten(); pr = pr.flatten()
    valid = (gt >= 0) & (gt < num_classes)
    gt = gt[valid]; pr = pr[valid]
    for i in range(num_classes):
        for j in range(num_classes):
            cm[i, j] += np.sum((gt == i) & (pr == j))
    return cm

def miou_pixacc_fwIoU_folder(gt_dir, pred_dir, num_classes=2):
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith('.tif')])
    cm_total = np.zeros((num_classes, num_classes), dtype=np.int64)
    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pred_dir, f)
        if not os.path.exists(pr_path):
            alt = f.replace('Img_','Mask_')
            pr_path = os.path.join(pred_dir, alt)
            if not os.path.exists(pr_path):
                continue
        gt = read_mask(gt_path); pr = read_mask(pr_path)
        cm_total += confusion_from_masks(gt, pr, num_classes)
    tp = np.diag(cm_total); total = cm_total.sum()
    pixacc = float(tp.sum()/total) if total>0 else 0.0
    ious=[]
    for c in range(num_classes):
        denom = tp[c] + (cm_total[c,:].sum()-tp[c]) + (cm_total[:,c].sum()-tp[c])
        ious.append(float(tp[c]/denom) if denom>0 else 0.0)
    miou = float(np.mean(ious)) if ious else 0.0
    freq = cm_total.sum(axis=1) / total if total>0 else np.zeros(num_classes)
    fwiou = float((freq * np.array(ious)).sum())
    return dict(mIoU=miou, pixAcc=pixacc, fwIoU=fwiou, IoUs=ious, CM=cm_total)

pix_PI = miou_pixacc_fwIoU_folder(GT_PI, PRED_PI_TH, num_classes=2)
pix_FW = miou_pixacc_fwIoU_folder(GT_FW, PRED_FW_TH, num_classes=2)

df_pix = pd.DataFrame([
    dict(Split="PI-test", mIoU=pix_PI["mIoU"], pixAcc=pix_PI["pixAcc"], fwIoU=pix_PI["fwIoU"]),
    dict(Split="FW-test", mIoU=pix_FW["mIoU"], pixAcc=pix_FW["pixAcc"], fwIoU=pix_FW["fwIoU"]),
])
try:
    display(df_pix)
except Exception:
    pass

df_pix.to_csv(f"{OUT_PI}/csv/pixel_metrics_theta_star.csv", index=False)
with open(f"{OUT_PI}/tables/table_pixel_metrics_theta_star.tex","w") as f:
    f.write(df_pix.to_latex(index=False, float_format='%.4f'))

print("Celda 8 lista: métricas pixelares a θ* guardadas.")

In [10]:
#@title 9. Cobertura/Densidad a θ* + WS calibrado (FW)
import os, math
import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage
from sklearn.metrics import mean_squared_error, r2_score

# ---- Utils mínimos ----
def read_mask(path):
    with rasterio.open(path) as src:
        return src.read(1)

# ---- WS helpers (skimage opcional; fallback a CC) ----
try:
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed
    from skimage.morphology import h_minima
    _SKIMAGE_OK = True
except Exception:
    _SKIMAGE_OK = False
    peak_local_max = watershed = h_minima = None
    print("scikit-image no disponible: usaré CC como fallback.")

def split_ws_pred(mask_bin, min_dist_px=3, h_rel=0.10):
    if not _SKIMAGE_OK:
        labels, _ = ndimage.label(mask_bin.astype(np.uint8))
        return labels
    from scipy import ndimage as ndi
    mask_bin = mask_bin.astype(np.uint8)
    dist = ndi.distance_transform_edt(mask_bin)
    if dist.max() > 0 and h_rel > 0:
        try:
            dist_supp = dist - h_minima(dist, h=float(h_rel * dist.max()))
        except Exception:
            dist_supp = dist
    else:
        dist_supp = dist
    coords = peak_local_max(dist_supp, min_distance=int(max(1, min_dist_px)), labels=mask_bin)
    markers = np.zeros_like(mask_bin, dtype=np.int32)
    for i, (r, c) in enumerate(coords, start=1):
        markers[r, c] = i
    labels = watershed(-dist_supp, markers, mask=mask_bin)
    return labels

def density_cc(mask_bin):
    _, n = ndimage.label(mask_bin.astype(np.uint8))
    return int(n)

def density_ws(mask_bin, min_dist_px=3, h_rel=0.10):
    labels = split_ws_pred(mask_bin.astype(np.uint8), min_dist_px=min_dist_px, h_rel=h_rel)
    return int(labels.max())

# ---- Cobertura y densidad por imagen (con NoData enmascarado) ----
def coverage_density_from_folders(gt_dir, pr_dir, ws_params=None):
    """
    Calcula cobertura (fracción 0–1) y densidad (conteo por imagen) para cada par GT/Pred.
    - Cobertura SOLO sobre píxeles válidos (NoData del GT).
    - Devuelve métricas de cobertura en **%** (RMSE/MAE/MBE) + R².
    - Devuelve métricas de densidad en unidades absolutas + R².
    - 'series' mantiene cov_T/cov_P en fracción (0–1) para figuras.
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    cov_T, cov_P, den_T, den_P = [], [], [], []

    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pr_dir, f)
        if not os.path.exists(pr_path):
            alt = f.replace("Img_", "Mask_")
            pr_path = os.path.join(pr_dir, alt)
            if not os.path.exists(pr_path):
                continue

        gt = read_mask(gt_path); pr = read_mask(pr_path)

        # Enmascarado NoData desde GT
        with rasterio.open(gt_path) as src:
            nod = src.nodata
        valid = (gt != nod) if nod is not None else np.ones_like(gt, dtype=bool)
        if valid.sum() == 0:
            continue

        gtb = (gt == 1) & valid
        prb = (pr == 1) & valid

        # Cobertura (0–1) sobre válidos
        cov_T.append(float(gtb.sum() / valid.sum()))
        cov_P.append(float(prb.sum() / valid.sum()))

        # Densidad (conteo por imagen)
        den_T.append(density_cc(gtb))
        if ws_params is None:
            den_P.append(density_cc(prb))
        else:
            den_P.append(density_ws(prb,
                                    min_dist_px=int(ws_params.get("min_dist_px", 3)),
                                    h_rel=float(ws_params.get("h_rel", 0.10))))

    # Cobertura en % (puntos porcentuales) + R²
    if cov_T:
        rmse_cover = float(np.sqrt(mean_squared_error(cov_T, cov_P))) * 100.0
        mae_cover  = float(np.mean(np.abs(np.array(cov_T) - np.array(cov_P)))) * 100.0
        mbe_cover  = float(np.mean(np.array(cov_P) - np.array(cov_T))) * 100.0
        r2_cover   = float(r2_score(cov_T, cov_P))
    else:
        rmse_cover = mae_cover = mbe_cover = r2_cover = float("nan")

    # Densidad en unidades absolutas + R²
    if den_T:
        rmse_density = float(np.sqrt(mean_squared_error(den_T, den_P)))
        mae_density  = float(np.mean(np.abs(np.array(den_T) - np.array(den_P))))
        mbe_density  = float(np.mean(np.array(den_P) - np.array(den_T)))
        r2_density   = float(r2_score(den_T, den_P))
    else:
        rmse_density = mae_density = mbe_density = r2_density = float("nan")

    out = dict(
        N=len(cov_T),
        RMSE_cover=rmse_cover, MAE_cover=mae_cover, MBE_cover=mbe_cover, R2_cover=r2_cover,
        RMSE_density=rmse_density, MAE_density=mae_density, MBE_density=mbe_density, R2_density=r2_density
    )
    return out, dict(cov_T=cov_T, cov_P=cov_P, den_T=den_T, den_P=den_P)

# ---- Área válida (ha) y densidad por hectárea ----
try:
    from pyproj import Geod
    _HAS_PYPROJ = True
    _GEOD = Geod(ellps="WGS84")
except Exception:
    _HAS_PYPROJ = False
    _GEOD = None

def _poly_area_m2_from_bounds(l, b, r, t):
    if not _HAS_PYPROJ:
        return None
    lons = [l, l, r, r, l]; lats = [b, t, t, b, b]
    area, _ = _GEOD.polygon_area_perimeter(lons, lats)
    return abs(area)

def _m_per_deg_lat(lat_rad):
    return (111132.92 - 559.82*math.cos(2*lat_rad) + 1.175*math.cos(4*lat_rad) - 0.0023*math.cos(6*lat_rad))

def _m_per_deg_lon(lat_rad):
    return (111412.84*math.cos(lat_rad) - 93.5*math.cos(3*lat_rad) + 0.118*math.cos(5*lat_rad))

def raster_valid_area_ha(img_path, valid_mask=None):
    with rasterio.open(img_path) as src:
        H, W = src.height, src.width
        tr, crs, bounds = src.transform, src.crs, src.bounds
        nodata = src.nodata
        if valid_mask is None:
            try:
                arr = src.read(1)
                valid_mask = (arr != nodata) if nodata is not None else np.ones((H, W), dtype=bool)
            except Exception:
                valid_mask = np.ones((H, W), dtype=bool)
        valid_px = int(np.sum(valid_mask))
        if valid_px == 0:
            return 0.0

        det = abs(tr.a * tr.e - tr.b * tr.d)
        if crs is not None and getattr(crs, "is_projected", False):
            return (det * valid_px) / 10000.0

        area_geo_m2 = _poly_area_m2_from_bounds(bounds.left, bounds.bottom, bounds.right, bounds.top)
        if area_geo_m2 is not None:
            frac = valid_px / float(H * W)
            return (area_geo_m2 * frac) / 10000.0

        # Aproximación métrica por grado
        lat_c = 0.5 * (bounds.bottom + bounds.top)
        lat_rad = math.radians(lat_c)
        mx = _m_per_deg_lon(lat_rad); my = _m_per_deg_lat(lat_rad)
        px_m2 = (mx * tr.a) * (my * abs(tr.e))
        return (valid_px * px_m2) / 10000.0

def eval_density_per_ha(gt_dir, pr_dir, ws_params=None):
    """
    Densidad (ind/ha), usando área válida (NoData enmascarado) y WS opcional.
    Devuelve y_true/y_pred + RMSE y R² en ind/ha.
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    yT, yP = [], []
    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pr_dir, f)
        if not os.path.exists(pr_path):
            alt = f.replace("Img_", "Mask_")
            pr_path = os.path.join(pr_dir, alt)
            if not os.path.exists(pr_path):
                continue

        gt = read_mask(gt_path); pr = read_mask(pr_path)
        with rasterio.open(gt_path) as src:
            nod = src.nodata
        valid = (gt != nod) if nod is not None else np.ones_like(gt, bool)
        area_ha = max(raster_valid_area_ha(gt_path, valid_mask=valid), 1e-9)

        gtb = (gt == 1) & valid
        prb = (pr == 1) & valid

        labs_gt, _ = ndimage.label(gtb.astype(np.uint8))
        dens_gt = int(labs_gt.max()) / area_ha

        if ws_params is None:
            labs_pr, _ = ndimage.label(prb.astype(np.uint8))
            dens_pr = int(labs_pr.max()) / area_ha
        else:
            labs_pr = split_ws_pred(prb.astype(np.uint8),
                                    min_dist_px=int(ws_params.get("min_dist_px", 3)),
                                    h_rel=float(ws_params.get("h_rel", 0.10)))
            dens_pr = int(labs_pr.max()) / area_ha

        yT.append(dens_gt); yP.append(dens_pr)

    rmse = float(np.sqrt(mean_squared_error(yT, yP))) if yT else np.nan
    r2   = float(r2_score(yT, yP)) if yT else np.nan
    return dict(y_true=yT, y_pred=yP, rmse=rmse, r2=r2)

# ---- Calibración WS por imagen (minimiza RMSE de conteo) ----
def calibrate_ws_density(gt_dir, pr_dir,
                         grid_min_dist=(2,3,4,5,6,7),
                         grid_h=(0.05,0.08,0.10,0.12,0.15,0.20)):
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    best = {"rmse": 1e9, "min_dist_px": None, "h_rel": None}
    for dmin in grid_min_dist:
        for h in grid_h:
            yT, yP = [], []
            for f in files:
                gt_path = os.path.join(gt_dir, f)
                pr_path = os.path.join(pr_dir, f)
                if not os.path.exists(pr_path):
                    alt = f.replace("Img_", "Mask_")
                    pr_path = os.path.join(pr_dir, alt)
                    if not os.path.exists(pr_path):
                        continue
                gt = read_mask(gt_path); pr = read_mask(pr_path)
                gtb = (gt == 1); prb = (pr == 1)
                labs_gt,_ = ndimage.label(gtb.astype(np.uint8)); dens_gt = int(labs_gt.max())
                labs_pr = split_ws_pred(prb.astype(np.uint8), min_dist_px=int(dmin), h_rel=float(h))
                dens_pr = int(labs_pr.max())
                yT.append(dens_gt); yP.append(dens_pr)
            if yT:
                rmse = float(np.sqrt(mean_squared_error(yT, yP)))
                if rmse < best["rmse"]:
                    best = {"rmse": rmse, "min_dist_px": int(dmin), "h_rel": float(h)}
    return best

# ================== Baseline + Calibración (usa tus rutas GT_* y PRED_* existentes) ==================
WS_PARAMS = dict(min_dist_px=3, h_rel=0.10)

# 1) Métricas por imagen (baseline)
pi_cd, _ = coverage_density_from_folders(GT_PI, PRED_PI_TH, ws_params=WS_PARAMS)
fw_cd, _ = coverage_density_from_folders(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS)
print("\nCobertura/Densidad (θ* | WS baseline) — PointRend INSTANCIAS")
print("PI: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    pi_cd["RMSE_cover"], pi_cd["MAE_cover"], pi_cd["MBE_cover"], pi_cd["R2_cover"],
    pi_cd["RMSE_density"], pi_cd["MAE_density"], pi_cd["MBE_density"], pi_cd["R2_density"]))
print("FW: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    fw_cd["RMSE_cover"], fw_cd["MAE_cover"], fw_cd["MBE_cover"], fw_cd["R2_cover"],
    fw_cd["RMSE_density"], fw_cd["MAE_density"], fw_cd["MBE_density"], fw_cd["R2_density"]))

# 2) Densidad por hectárea (baseline)
pi_ha = eval_density_per_ha(GT_PI, PRED_PI_TH, ws_params=WS_PARAMS)
fw_ha = eval_density_per_ha(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS)
print("\nDensidad por hectárea (θ* | WS baseline)")
print("PI: n={} | RMSE={:.2f} ind/ha | R²={:.3f}".format(len(pi_ha["y_true"]), pi_ha["rmse"], pi_ha["r2"]))
print("FW: n={} | RMSE={:.2f} ind/ha | R²={:.3f}".format(len(fw_ha["y_true"]), fw_ha["rmse"], fw_ha["r2"]))

# 3) Calibración WS (FW) sobre PRED_FW_TH
best_ws_fw = calibrate_ws_density(GT_FW, PRED_FW_TH)
print("\nWS FW óptimo:", best_ws_fw)
WS_PARAMS_FW = dict(min_dist_px=int(best_ws_fw["min_dist_px"]), h_rel=float(best_ws_fw["h_rel"]))

fw_cd_cal, _ = coverage_density_from_folders(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS_FW)
fw_ha_cal = eval_density_per_ha(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS_FW)

print("\nCobertura/Densidad FW (θ* + WS calibrado)")
print("FW: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    fw_cd_cal["RMSE_cover"], fw_cd_cal["MAE_cover"], fw_cd_cal["MBE_cover"], fw_cd_cal["R2_cover"],
    fw_cd_cal["RMSE_density"], fw_cd_cal["MAE_density"], fw_cd_cal["MBE_density"], fw_cd_cal["R2_density"]))
print("\nDensidad por hectárea FW (θ* + WS calibrado)")
print("FW: n={} | RMSE={:.2f} ind/ha | R²={:.3f}".format(len(fw_ha_cal["y_true"]), fw_ha_cal["rmse"], fw_ha_cal["r2"]))

# 4) Guardados (CSV/LaTeX) con sufijo PointRendInst
os.makedirs(f"{OUT_PI}/csv", exist_ok=True)
os.makedirs(f"{OUT_PI}/tables", exist_ok=True)
df_covdens = pd.DataFrame([
    dict(Split="PI-test (PointRend-Inst)", **pi_cd, RMSE_dens_ha=pi_ha["rmse"], R2_dens_ha=pi_ha["r2"]),
    dict(Split="FW-test (PointRend-Inst, WS base)", **fw_cd, RMSE_dens_ha=fw_ha["rmse"], R2_dens_ha=fw_ha["r2"]),
    dict(Split="FW-test (PointRend-Inst, WS cal)", **fw_cd_cal, RMSE_dens_ha=fw_ha_cal["rmse"], R2_dens_ha=fw_ha_cal["r2"]),
])
csv_path = f"{OUT_PI}/csv/coverage_density_theta_star_PointRendInst.csv"
tex_path = f"{OUT_PI}/tables/table_coverage_density_theta_star_PointRendInst.tex"
df_covdens.to_csv(csv_path, index=False)
with open(tex_path, "w") as f:
    f.write(df_covdens.to_latex(index=False, float_format="%.4f"))
print("\nGuardado resumen PointRend INSTANCIAS en:")
print(" CSV  ->", csv_path)
print(" LaTeX->", tex_path)


Cobertura/Densidad (θ* | WS baseline) — PointRend INSTANCIAS
PI: RMSE_cover=1.48%  MAE_cover=0.65%  MBE_cover=-0.41%  |  R²_cover=0.9666  ||  RMSE_dens=25.34  MAE_dens=12.56  MBE_dens=+12.51  R²_dens=-5.588
FW: RMSE_cover=4.49%  MAE_cover=1.90%  MBE_cover=-1.63%  |  R²_cover=0.8351  ||  RMSE_dens=29.82  MAE_dens=15.27  MBE_dens=+14.95  R²_dens=-2.206

Densidad por hectárea (θ* | WS baseline)
PI: n=75 | RMSE=71.16 ind/ha | R²=-5.591
FW: n=124 | RMSE=117.80 ind/ha | R²=-2.207

WS FW óptimo: {'rmse': 6.9740301672844485, 'min_dist_px': 7, 'h_rel': 0.05}

Cobertura/Densidad FW (θ* + WS calibrado)
FW: RMSE_cover=4.49%  MAE_cover=1.90%  MBE_cover=-1.63%  |  R²_cover=0.8351  ||  RMSE_dens=6.97  MAE_dens=3.91  MBE_dens=+2.07  R²_dens=0.825

Densidad por hectárea FW (θ* + WS calibrado)
FW: n=124 | RMSE=27.56 ind/ha | R²=0.824

Guardado resumen PointRend INSTANCIAS en:
 CSV  -> /content/drive/MyDrive/juniper_mapper/JuniperMapper/PointRend_instance/output_PI/csv/coverage_density_theta_star_PointR

In [ ]:
#@title 10. Figuras extra — Scatter FW y Barras por tamaño (PI/FW) @ θ*
from scipy.stats import pearsonr

# Series FW para scatter (usando WS calibrado si existe, si no baseline)
try:
    _WS = WS_PARAMS_FW
except NameError:
    try:
        _WS = WS_PARAMS
    except NameError:
        _WS = dict(min_dist_px=3, h_rel=0.10)

_, _series_fw = coverage_density_from_folders(GT_FW, PRED_FW_TH, ws_params=_WS)
_cov_T = np.array(_series_fw["cov_T"]) * 100.0
_cov_P = np.array(_series_fw["cov_P"]) * 100.0
_den_fw = eval_density_per_ha(GT_FW, PRED_FW_TH, ws_params=_WS)
_den_T = np.array(_den_fw["y_true"]) ; _den_P = np.array(_den_fw["y_pred"])

def _rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2))) if len(y_true) else float("nan")

def _mae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred))) if len(y_true) else float("nan")

def _mbe(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(y_pred - y_true)) if len(y_true) else float("nan")

plt.rcParams.update({"font.size":11,"axes.titlesize":12,"axes.labelsize":11,"legend.fontsize":10,"figure.dpi":150})
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# (a) Cobertura
r_cov = pearsonr(_cov_T, _cov_P)[0] if len(_cov_T)>1 else np.nan
axs[0].scatter(_cov_T, _cov_P, s=12)
lim = [0, max(1e-6, _cov_T.max() if len(_cov_T)>0 else 0, _cov_P.max() if len(_cov_P)>0 else 0) * 1.05]
axs[0].plot(lim, lim, linestyle="--")
axs[0].set_xlim(lim); axs[0].set_ylim(lim)
axs[0].set_xlabel("Observed canopy cover (%)"); axs[0].set_ylabel("Predicted canopy cover (%)")
axs[0].set_title(f"(a) FW — Cover @ θ={theta_fw:.2f}")
axs[0].text(
    0.02, 0.98,
    f"r={r_cov:.2f}\\nRMSE={_rmse(_cov_T,_cov_P):.2f}%\\nMAE={_mae(_cov_T,_cov_P):.2f}%\\nMBE={_mbe(_cov_T,_cov_P):.2f}",
    transform=axs[0].transAxes, va="top"
)

# (b) Densidad
r_den = pearsonr(_den_T, _den_P)[0] if len(_den_T)>1 else np.nan
axs[1].scatter(_den_T, _den_P, s=12)
lim2 = [0, max(1e-6, _den_T.max() if len(_den_T)>0 else 0, _den_P.max() if len(_den_P)>0 else 0) * 1.05]
axs[1].plot(lim2, lim2, linestyle="--")
axs[1].set_xlim(lim2); axs[1].set_ylim(lim2)
axs[1].set_xlabel("Observed shrubs per ha"); axs[1].set_ylabel("Predicted shrubs per ha")
axs[1].set_title(f"(b) FW — Density @ θ={theta_fw:.2f}")
axs[1].text(
    0.02, 0.98,
    f"r={r_den:.2f}\\nRMSE={_rmse(_den_T,_den_P):.2f}\\nMAE={_mae(_den_T,_den_P):.2f}\\nMBE={_mbe(_den_T,_den_P):.2f}",
    transform=axs[1].transAxes, va="top"
)

fig.suptitle("Observed vs Predicted — FW — PointRend Inst")
fig.tight_layout(rect=[0, 0, 1, 0.93])
scatter_fw = f"{OUT_PI}/plots/FW_scatter_cover_density_PointRendInst.png"
plt.savefig(scatter_fw, bbox_inches="tight"); plt.close()
print("Scatter FW guardado en:", scatter_fw)

# Barras por tamaño usando df_pi_sz / df_fw_sz generados en Celda 7

def _plot_sizewise_bars_point(dfsz, split_name, out_name, theta_txt):
    order = [b[0] for b in SIZE_BINS] + ["All"]
    dfsz = dfsz.set_index("Size").reindex(order).reset_index()
    labels = dfsz["Size"].tolist()
    x = np.arange(len(labels)); width = 0.38
    plt.rcParams.update({"font.size":11,"axes.titlesize":12,"axes.labelsize":11,"legend.fontsize":10,"figure.dpi":150})
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    ax.bar(x - width/2, dfsz["IoU_F1"].values, width, label="IoU @ 0.5")
    ax.bar(x + width/2, dfsz["S-IoU_F1"].values, width, label="S-IoU @ 0.5")
    ax.set_xticks(x, labels); ax.set_ylabel("F1-score (%)"); ax.set_xlabel("Size bin")
    ax.set_title(f"{split_name} — Size-wise F1 (θ={theta_txt}) — PointRend Inst")
    ax.grid(axis="y", alpha=0.3); ax.legend(loc="best", frameon=False)
    outp = f"{OUT_PI}/plots/{out_name}"
    plt.savefig(outp, bbox_inches="tight"); plt.close()
    return outp

pi_bars = _plot_sizewise_bars_point(df_pi_sz, "PI", "PI_sizewise_bars_PointRendInst.png", f"{theta_pi:.2f}")
fw_bars = _plot_sizewise_bars_point(df_fw_sz, "FW", "FW_sizewise_bars_PointRendInst.png", f"{theta_fw:.2f}")
print("Barras por tamaño guardadas en: \\n-", pi_bars, " \\n -", fw_bars)